# LangGraph: Nodes, Edges & State
**Module 5, Topic 2 — Automation & Agentic Systems**

## The Problem

In Topic 1 we built a tool-calling loop — a straight line: question → tool → answer. But real workflows have **decision points**: if a NOC alert is P1, escalate immediately; if P3, just log it. Nested `if/else` chains get brittle fast, and you can't visualise or test the flow.

## The Solution

**LangGraph** lets you express branching workflows as a **graph** — nodes are processing steps, edges are transitions, and conditional edges route based on state. The graph is inspectable, testable, and composable.

In this notebook you will:
1. Define a shared **State** that every node reads and writes
2. Write **nodes** (plain Python functions) that call an LLM and return state updates
3. Connect nodes with **edges** and **conditional edges**
4. Trace state through the graph for debugging
5. Use **annotated state** to append to lists instead of replacing them


## Setup

In [ ]:
!pip install langgraph openai python-dotenv -q

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("MODEL_NAME")

print(f"Client ready — model: {MODEL}")

Client ready — model: google/gemma-4-E26B-it


## Step 1 — Define the State

State is a `TypedDict` that every node reads and writes.  
Each node returns a dict of fields to update — LangGraph merges them in.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class AlertState(TypedDict):
    alert_text: str
    severity: str
    action: str
    response: str


print("State schema:", AlertState.__annotations__)

State schema: {'alert_text': <class 'str'>, 'severity': <class 'str'>, 'action': <class 'str'>, 'response': <class 'str'>}


## Step 2 — Write the Nodes

Each node is a plain Python function.  
- `triage` — calls the LLM to classify severity  
- `escalate` — builds an escalation message for P1/P2  
- `log_and_notify` — builds a log entry for P3/P4

In [3]:
def triage(state: AlertState) -> dict:
    """Ask the LLM to classify the alert as P1, P2, P3, or P4."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a NOC triage system. Read the alert and respond with "
                    "exactly one token: P1, P2, P3, or P4. Nothing else."
                ),
            },
            {"role": "user", "content": state["alert_text"]},
        ],
    )
    severity = response.choices[0].message.content.strip()
    return {"severity": severity}


print("triage node defined")

triage node defined


In [4]:
def escalate(state: AlertState) -> dict:
    """Draft an escalation message for high-severity alerts."""
    return {
        "action": "escalated",
        "response": (
            f"ESCALATION: {state['severity']} alert forwarded to duty manager.\n"
            f"Alert: {state['alert_text'][:120]}..."
        ),
    }


def log_and_notify(state: AlertState) -> dict:
    """Log the alert and prepare a team notification."""
    return {
        "action": "logged",
        "response": (
            f"Logged: {state['severity']} alert. Team notified by email.\n"
            f"Alert: {state['alert_text'][:120]}..."
        ),
    }


print("escalate and log_and_notify nodes defined")

escalate and log_and_notify nodes defined


## Step 3 — Build the Graph

Add the nodes, then connect them with edges.  
The triage node needs a **conditional edge** — the next node depends on the severity.

In [5]:
def route_by_severity(state: AlertState) -> str:
    """Choose the next node based on severity."""
    if state["severity"] in ("P1", "P2"):
        return "escalate"
    return "log_and_notify"


# Build the graph
graph = StateGraph(AlertState)

graph.add_node("triage", triage)
graph.add_node("escalate", escalate)
graph.add_node("log_and_notify", log_and_notify)

graph.add_edge(START, "triage")
graph.add_conditional_edges(
    "triage",
    route_by_severity,
    {"escalate": "escalate", "log_and_notify": "log_and_notify"},
)
graph.add_edge("escalate", END)
graph.add_edge("log_and_notify", END)

app = graph.compile()

print("Graph compiled successfully")

Graph compiled successfully


## Step 4 — Run a High-Severity Alert

A carrier loss affecting enterprise customers — should be classified as P1 or P2 and escalated.

In [6]:
result = app.invoke({
    "alert_text": (
        "Loss of carrier on KU-HUB1-GW3. 3 enterprise customers affected. "
        "41 terminals offline. Severity appears high."
    ),
    "severity": "",
    "action": "",
    "response": "",
})

print(f"Severity : {result['severity']}")
print(f"Action   : {result['action']}")
print(f"Response : {result['response']}")

Severity : P1
Action   : escalated
Response : ESCALATION: P1 alert forwarded to duty manager.
Alert: Loss of carrier on KU-HUB1-GW3. 3 enterprise customers affected. 41 terminals offline. Severity appears high....


## Step 5 — Run a Low-Severity Alert

A routine maintenance notification — should be classified as P3 or P4 and logged.

In [ ]:
result_low = app.invoke({
    "alert_text": (
        "Routine maintenance notification: scheduled firmware upgrade on "
        "backup link KA-BKP2-WEST. No customer impact expected. "
        "Window: 02:00-04:00 WAT."
    ),
    "severity": "",
    "action": "",
    "response": "",
})

print(f"Severity : {result_low['severity']}")
print(f"Action   : {result_low['action']}")
print(f"Response : {result_low['response']}")

## Step 6 — Tracing the State

Wrap each node so it prints its state updates — this is how you debug a graph.

In [ ]:
def traced(fn):
    """Wrapper that prints state updates from a node."""
    def wrapper(state):
        updates = fn(state)
        print(f"[{fn.__name__}] returned: {updates}")
        return updates
    wrapper.__name__ = fn.__name__
    return wrapper


# Rebuild the graph with traced nodes
traced_graph = StateGraph(AlertState)

traced_graph.add_node("triage", traced(triage))
traced_graph.add_node("escalate", traced(escalate))
traced_graph.add_node("log_and_notify", traced(log_and_notify))

traced_graph.add_edge(START, "triage")
traced_graph.add_conditional_edges(
    "triage",
    route_by_severity,
    {"escalate": "escalate", "log_and_notify": "log_and_notify"},
)
traced_graph.add_edge("escalate", END)
traced_graph.add_edge("log_and_notify", END)

traced_app = traced_graph.compile()

print("Traced graph compiled — run it to see state updates\n")

result_traced = traced_app.invoke({
    "alert_text": (
        "Intermittent packet loss on C-BAND-LINK-7. One government client "
        "reporting degraded video conferencing. Duration: 45 minutes."
    ),
    "severity": "",
    "action": "",
    "response": "",
})

print(f"\nFinal state: severity={result_traced['severity']}, action={result_traced['action']}")

## Step 7 — Annotated State (Appending to Lists)

By default, returning a field replaces it. For fields like logs or messages, you want to **append**.  
`Annotated[list, add]` tells LangGraph to concatenate instead of replace.

In [ ]:
from typing import Annotated
from operator import add


class LoggingState(TypedDict):
    alert_text: str
    severity: str
    action: str
    response: str
    log: Annotated[list, add]  # each node appends to this list


def triage_with_log(state: LoggingState) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a NOC triage system. Read the alert and respond with "
                    "exactly one token: P1, P2, P3, or P4. Nothing else."
                ),
            },
            {"role": "user", "content": state["alert_text"]},
        ],
    )
    severity = response.choices[0].message.content.strip()
    return {"severity": severity, "log": [f"triage: classified as {severity}"]}


def escalate_with_log(state: LoggingState) -> dict:
    return {
        "action": "escalated",
        "response": f"ESCALATION: {state['severity']} alert forwarded to duty manager.",
        "log": ["escalate: forwarded to duty manager"],
    }


def log_and_notify_with_log(state: LoggingState) -> dict:
    return {
        "action": "logged",
        "response": f"Logged: {state['severity']} alert. Team notified.",
        "log": ["log_and_notify: logged and team emailed"],
    }


def route_logged(state: LoggingState) -> str:
    if state["severity"] in ("P1", "P2"):
        return "escalate"
    return "log_and_notify"


log_graph = StateGraph(LoggingState)
log_graph.add_node("triage", triage_with_log)
log_graph.add_node("escalate", escalate_with_log)
log_graph.add_node("log_and_notify", log_and_notify_with_log)

log_graph.add_edge(START, "triage")
log_graph.add_conditional_edges(
    "triage",
    route_logged,
    {"escalate": "escalate", "log_and_notify": "log_and_notify"},
)
log_graph.add_edge("escalate", END)
log_graph.add_edge("log_and_notify", END)

log_app = log_graph.compile()

result_log = log_app.invoke({
    "alert_text": (
        "Total loss of signal on KU-HUB2-NORTH. 12 banking terminals offline. "
        "Immediate attention required."
    ),
    "severity": "",
    "action": "",
    "response": "",
    "log": [],
})

print("Log entries (appended, not replaced):")
for entry in result_log["log"]:
    print(f"  - {entry}")

## Summary

You built a LangGraph graph with:
- **State** — a TypedDict shared by every node
- **Nodes** — plain Python functions that read and update state
- **Edges** — fixed connections between nodes
- **Conditional edges** — a routing function that picks the next node based on state
- **Annotated state** — `Annotated[list, add]` for fields that grow instead of being replaced

In Topic 3 we use this same pattern to build the compliance-checker agent with tools, retrieval, and a loop.

## Exercises

### Exercise 1 — Add a new severity branch
Add a third handler node called `auto_resolve` for P4 alerts that returns `{"action": "auto_resolved", ...}`. Update `route_by_severity` so P3 goes to `log_and_notify` and P4 goes to `auto_resolve`. Test with the maintenance alert from Step 5.

### Exercise 2 — Add a `reviewed_by` field
Add a `reviewed_by: str` field to `AlertState`. Write a new node `assign_reviewer` that sets it to `"duty_manager"` for P1 or `"shift_lead"` for P2/P3. Insert it between triage and the escalate/log branches. You will need to restructure the conditional edge.

### Exercise 3 — Trace the annotated-state graph
Apply the `traced()` wrapper from Step 6 to the `LoggingState` graph from Step 7. Run it and confirm that the `log` list grows through each node rather than being replaced.

**Time:** 20 minutes

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here
